In [3]:
import numpy as np
import time
import sys
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Ajuste de segurança para a pilha recursiva dos QuickSorts
sys.setrecursionlimit(30000)

# =====================================================================
# CORE: ALGORITMOS DE ORDENAÇÃO EXIGIDOS NO PROJETO
# =====================================================================

def insertionSort(V):
    N = len(V)
    for j in range(1, N):
        key = V[j]
        i = j - 1
        while i >= 0 and V[i] > key:
            V[i + 1] = V[i]
            i -= 1
        V[i + 1] = key

def shellSort(V, gaps):
    N = len(V)
    for h in gaps:
        for i in range(h, N):
            key = V[i]
            j = i
            while j >= h and V[j - h] > key:
                V[j] = V[j - h]
                j -= h
            V[j] = key

def partitionHoare(V, i, f):
    p = V[i]
    a = i - 1    
    b = f + 1    
    while True:
        while True:
            a += 1
            if V[a] >= p: 
                break 
        while True:
            b -= 1
            if V[b] <= p: 
                break 
        if a >= b: 
            return b
        V[a], V[b] = V[b], V[a]

def quickSortHoare(V, i, f):
    if i < f:
        pivot = partitionHoare(V, i, f)
        if pivot == f:
            quickSortHoare(V, i, pivot - 1)
        else:
            quickSortHoare(V, i, pivot)
            quickSortHoare(V, pivot + 1, f)

def partition3(A, i, f): 
    menors = i    
    x = i        
    maiors = f    
    pivot = A[i]  
    while x <= maiors:
        if A[x] < pivot:
            A[menors], A[x] = A[x], A[menors]
            menors += 1
            x += 1
        elif A[x] > pivot:
            A[x], A[maiors] = A[maiors], A[x]
            maiors -= 1
        else:
            x += 1
    return menors, maiors

def quickSort3Particoes(A, i, f):
    if i < f:
        esq, dir_idx = partition3(A, i, f)
        quickSort3Particoes(A, i, esq - 1)
        quickSort3Particoes(A, dir_idx + 1, f)

# =====================================================================
# INTERFACE EM BOTÕES: EXECUÇÃO AUTOMATIZADA COM 15 ITERAÇÕES
# =====================================================================

resultados_benchmark = {}

def executar_experimento(num_execucoes=15):
    """
    Executa o benchmark colhendo a estatística de tendência central (mediana)
    e dispersão (desvio padrão) ao longo de 15 execuções por cada tamanho N.
    """
    rng = np.random.default_rng()
    
    # Definição dos tamanhos N para validação em sala de aula
    tamanhos = np.linspace(30, 2000, num=8, dtype=int) 
    
    algoritmos = {
        "Insertion Sort": lambda v: insertionSort(v),
        "Shell Sort": lambda v: shellSort(v, [len(v)//(2**i) for i in range(1, int(np.log2(len(v)))+1) if len(v)//(2**i) > 0]),
        "QS Hoare": lambda v: quickSortHoare(v, 0, len(v) - 1),
        "QS 3 Partições": lambda v: quickSort3Particoes(v, 0, len(v) - 1)
    }
    
    dados_finais = {alg: {N: [] for N in tamanhos} for alg in algoritmos}
    
    # População Teórica Base (Distribuição Normal)
    populacao_normal = rng.normal(loc=10, scale=2, size=10000)
    
    print(f"▶️ A iniciar recolha estatística de dados ({num_execucoes} execuções independentes por tamanho)...")
    
    for it in range(num_execucoes):
        print(f"   A processar iteração de estabilização temporal {it+1}/{num_execucoes}...", end="\r")
        for N in tamanhos:
            amostra_continua = rng.choice(populacao_normal, size=N, replace=False)
            amostra_base = np.round(amostra_continua * 10).astype(int)
            
            min_val = np.min(amostra_base)
            if min_val <= 0:
                amostra_base = amostra_base - min_val + 1
                
            for nome_alg, func_alg in algoritmos.items():
                v_teste = amostra_base.copy()
                
                t_ini = time.perf_counter()
                func_alg(v_teste)
                t_fim = time.perf_counter()
                
                dados_finais[nome_alg][N].append(t_fim - t_ini)
                
    print("\n✅ Recolha terminada com sucesso absoluto!")
    
    resumo = {alg: {"tamanhos": tamanhos, "medianas": [], "desvios": []} for alg in algoritmos}
    for nome_alg in algoritmos:
        for N in tamanhos:
            tempos = dados_finais[nome_alg][N]
            resumo[nome_alg]["medianas"].append(np.median(tempos))
            resumo[nome_alg]["desvios"].append(np.std(tempos))
            
    return resumo

# =====================================================================
# REGRAS DE VISUALIZAÇÃO GRÁFICA (PLOTS)
# =====================================================================

def plotar_tendencia_central(resumo):
    plt.figure(figsize=(10, 6))
    for nome_alg, dados in resumo.items():
        plt.plot(dados["tamanhos"], dados["medianas"], marker='o', label=f"{nome_alg} (Mediana)")
        
    plt.title("Estudo de Complexidade Temporal: Caso Médio (Mediana de 15 Execuções)")
    plt.xlabel("Tamanho do Vetor (N)")
    plt.ylabel("Tempo de Execução (Segundos)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.show()

def plotar_dispersao(resumo):
    fig, axs = plt.subplots(2, 2, figsize=(12, 10), sharex=True)
    axs = axs.ravel()
    
    for idx, (nome_alg, dados) in enumerate(resumo.items()):
        axs[idx].errorbar(dados["tamanhos"], dados["medianas"], yerr=dados["desvios"], 
                          fmt='-o', color='purple', ecolor='red', capsize=4, label='Desvio Padrão')
        axs[idx].set_title(f"Dispersão Temporal: {nome_alg}")
        axs[idx].set_ylabel("Tempo (Segundos)")
        axs[idx].grid(True, linestyle=":", alpha=0.5)
        axs[idx].legend()
        
    plt.xlabel("Tamanho do Vetor (N)")
    plt.tight_layout()
    plt.show()

# =====================================================================
# INTERFACE INTERATIVA WIDGETS
# =====================================================================

btn_executar = widgets.Button(description="1. Rodar Benchmark (15x)", button_style="primary", icon="play")
btn_plot_central = widgets.Button(description="2. Plot Tendência Central", button_style="success", icon="chart-line", disabled=True)
btn_plot_dispersao = widgets.Button(description="3. Plot Barras de Dispersão", button_style="info", icon="chart-bar", disabled=True)

saida_graficos = widgets.Output()

def ao_clicar_executar(b):
    global resultados_benchmark
    with saida_graficos:
        clear_output()
        resultados_benchmark = executar_experimento(num_execucoes=15)
        btn_plot_central.disabled = False
        btn_plot_dispersao.disabled = False

def ao_clicar_central(b):
    with saida_graficos:
        clear_output()
        plotar_tendencia_central(resultados_benchmark)

def ao_clicar_dispersao(b):
    with saida_graficos:
        clear_output()
        plotar_dispersao(resultados_benchmark)

btn_executar.on_click(ao_clicar_executar)
btn_plot_central.on_click(ao_clicar_central)
btn_plot_dispersao.on_click(ao_clicar_dispersao)

caixa_botoes = widgets.HBox([btn_executar, btn_plot_central, btn_plot_dispersao])
display(caixa_botoes, saida_graficos)

Output()